# OCVWorkChain: r2SCAN 

Runs the **entire** OCV workflow with the r2SCAN functional

Requires aiida-open_circuit_voltage >= 0.7, aiida-quantumespresso >= 5.0, and Quantum ESPRESSO >= 7.6

Notes before using this route vs using r2SCAN//PBEsol:

- QE 7.6 is the only version implementing spin-polarised meta-GGA stress (pw.x 7.4/7.5 abort in `stres_gradcorr`), and it needs to be compiled with LibXC (r2SCAN is addressed through the `input_dft` string) and a norm-conserving pseudopotential family (`PseudoDojo/0.4/PBEsol/SR/stringent/upf` via `aiida-pseudo install pseudo-dojo`). 

- Meta-GGA variable-cell relaxations at norm-conserving cutoffs are substantially more expensive than GGA. 
The two-stage r2SCAN//PBEsol workflow in `Submit_r2SCAN.ipynb` (PBEsol relaxations + r2SCAN single-point energies) gives good enough voltages

## Loading libraries

In [ ]:
from aiida import load_profile, orm
## Indicate your profile name here
your_profile_name = 'cathodes'
load_profile(your_profile_name)
from aiida.plugins import WorkflowFactory
from aiida.engine import submit
import time
import pandas as pd

## Code, constants, structures and group

In [ ]:
## Code and constants
## Full r2SCAN runs on QE 7.6 (the only version with spin-polarised meta-GGA stress).
code = orm.load_code('pw_qe-7.6@alps')

## Daint GH200: 1 MPI rank per GPU, 4 GPUs/node; the `alps` computer prepend pins
## --ntasks-per-node=4 --cpus-per-task=71 --gpus-per-task=1; npool = #GPUs. Normal partition: 24 h.
time_in_s, num_machines, num_mpiprocs_per_machine, num_cores_per_mpiproc, npool = 86200, 1, 4, 1, 4

R2SCAN_PSEUDO_FAMILY = 'PseudoDojo/0.4/PBEsol/SR/stringent/upf'   # norm-conserving (meta-GGA requirement)
R2SCAN_FUNCTIONAL = 'XC-000I-000I-000I-000I-497L-498L'            # r2SCAN via LibXC
CATION = 'Li'                                                     # shared by all structures below
## r2SCAN bulk-Li reference (eV/atom, smearing-corrected): r2SCAN vc-relax of bulk Li at QE 7.6.
## Supplied via DFT_energy_bulk_Li below, so no bulk SCF is launched.
R2SCAN_BULK_LI_ENERGY = -199.20742349305587

## Structures to run
structure = orm.load_node('096d9d96-7f66-442b-a27f-2660572808ea') # LiCoO2

group_label = "ocv_r2scan_full_potential_cathodes"

## The submission function

In [ ]:
## The full-r2SCAN submission, wrapped as a function
import copy

OCVWorkChain = WorkflowFactory('quantumespresso.ocv.ocvwc')

def launch_r2scan_ocv(structure):
    """Build and submit one full-r2SCAN OCVWorkChain for `structure`; return the workchain node."""
    r2scan_pw = {'pseudo_family': R2SCAN_PSEUDO_FAMILY,
                 ## forc_conv_thr 2e-4: the protocol defaults sit below the meta-GGA force-noise
                 ## floor and the relax never converges
                 'pw': {'parameters': {'CONTROL':   {'forc_conv_thr': 2.0e-4},
                                       'SYSTEM':    {'input_dft': R2SCAN_FUNCTIONAL},
                                       'ELECTRONS': {'mixing_beta': 0.3}},
                        'parallelization': {'npool': npool}}}

    overrides = {
        'ocv_parameters': {
            'cation': CATION,
            ## no `for_r2scan` needed on this path
            'do_low_SOC_OCV': False,    # set False for the average voltage only (much cheaper)
            'do_high_SOC_OCV': False,
            ## r2SCAN volume shift can be higher than GGA
            'volume_change_stability_threshold': 0.15,
            'DFT_energy_bulk_Li': R2SCAN_BULK_LI_ENERGY
        },
        'ocv_relax': {'base_relax': copy.deepcopy(r2scan_pw)},
    }
    ## Optional: meta-GGA kinetic-energy densities converge slowly with the charge-density cutoff;
    ## the PseudoDojo default dual is 4, use following to run with dual 8:
    # ecutwfc, _ = orm.load_group(R2SCAN_PSEUDO_FAMILY).get_recommended_cutoffs(structure=structure, unit='Ry')
    # overrides['ocv_relax']['base_relax']['pw']['parameters']['SYSTEM']['ecutrho'] = 8 * ecutwfc

    ## bulk_cation_structure is deliberately NOT passed, otherwise the workchain would run its own
    ## bulk SCF and IGNORE DFT_energy_bulk_Li
    builder = OCVWorkChain.get_builder_from_protocol(code=code, structure=structure, protocol='balanced', overrides=overrides)
    builder.clean_workdir = orm.Bool(False)

    pw_dict = builder.ocv_relax.base_relax.pw
    pw_dict.metadata['options']['max_wallclock_seconds'] = time_in_s
    pw_dict.metadata['options']['resources']['num_machines'] = num_machines
    pw_dict.metadata['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
    pw_dict.metadata['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
    pw_dict.parameters['ELECTRONS']['electron_maxstep'] = 200

    return submit(builder)

## Launching over all structures

In [ ]:
## Launching one workchain per structure, collected in the group
group, created = orm.Group.collection.get_or_create(group_label)
print(f"{'Created' if created else 'Reusing'} group '{group.label}' (PK={group.pk})")

node = launch_r2scan_ocv(structure)
group.add_nodes(node)
print(f'{structure.get_formula()}: submitted full-r2SCAN OCVWorkChain PK={node.pk}')

## Results

In [ ]:
for node in orm.load_group(group_label).nodes:
    formula = node.inputs.structure.get_formula()
    if node.is_finished_ok:
        print(formula, node.pk, node.outputs.open_circuit_voltages.get_dict())
    else:
        print(formula, node.pk, node.process_state.value, node.exit_status)
node.outputs.common_workflow_output   # full JSON (voltages + structures) of one workchain